# DataSet Football Value Prediction


In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. โหลดข้อมูล (อย่าลืมอัปโหลดไฟล์ csv เข้า Colab ก่อน)
stats = pd.read_csv('all_player_stats.csv')
profiles = pd.read_csv('all_player_profiles.csv')
df = pd.merge(profiles, stats.drop(columns=['league']), on='player_id').dropna(subset=['market_value'])

# 2. เลือก Features ให้ตรงกับหน้าเว็บ
X = df[['league', 'position', 'appearances', 'goals', 'assists', 'rating', 'minutes_played', 'tackles']]
y = df['market_value']

# 3. สร้าง Preprocessor (กำหนด remainder='drop' เพื่อแก้ Error _RemainderColsList)
num_features = ['appearances', 'goals', 'assists', 'rating', 'minutes_played', 'tackles']
cat_features = ['league', 'position']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('std', StandardScaler())]), num_features),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
    ],
    remainder='drop'
)

# 4. สร้าง Ensemble Model
ensemble = VotingRegressor([
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingRegressor(random_state=42)),
    ('ridge', Ridge())
])

# 5. เทรนและบันทึก
model = Pipeline([('pre', preprocessor), ('model', ensemble)])
model.fit(X, y)

joblib.dump(model, 'football_ml_model.joblib', compress=3)
print("✅ บันทึก football_ml_model.joblib เรียบร้อย!")

✅ บันทึก football_ml_model.joblib เรียบร้อย!


#DataSet Wine Quality Grading




In [ ]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from tensorflow.keras import layers, models

# 1. โหลดข้อมูล
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df_wine = pd.read_csv(url, sep=';')
X_w = df_wine.drop('quality', axis=1)
y_w = df_wine['quality']

# 2. Preprocessing
wine_imputer = SimpleImputer(strategy='mean')
wine_scaler = StandardScaler()
X_prepped = wine_scaler.fit_transform(wine_imputer.fit_transform(X_w))

# 3. สร้างโมเดล NN
nn = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_prepped.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(11, activation='softmax')
])

nn.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
nn.fit(X_prepped, y_w, epochs=30, verbose=0)

# --- จุดสำคัญ: วิธีเซฟเพื่อให้เครื่องเวอร์ชันเก่าอ่านได้ ---

# --- ใน Colab ตอนบันทึกโมเดล ---

# ก. เซฟเฉพาะ "น้ำหนัก" (Weights) - ต้องเติม .weights นำหน้า h5
nn.save_weights('wine_nn_weights.weights.h5')

# ข. เซฟ "โครงสร้าง" เป็น JSON
model_json = nn.to_json()
with open("wine_model_config.json", "w") as json_file:
    json_file.write(model_json)

# ค. เซฟ Scaler และ Imputer (เหมือนเดิม)
joblib.dump(wine_scaler, 'wine_scaler.joblib')
joblib.dump(wine_imputer, 'wine_imputer.joblib')

print("✅ ดาวน์โหลดไฟล์: wine_nn_weights.h5 และ wine_model_config.json ไปใช้แทนไฟล์เดิม")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


✅ ดาวน์โหลดไฟล์: wine_nn_weights.h5 และ wine_model_config.json ไปใช้แทนไฟล์เดิม


# ดูVersion Library

In [ ]:
import pandas
import joblib
import sklearn
import tensorflow
import numpy

print(f"pandas version: {pandas.__version__}")
print(f"joblib version: {joblib.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"tensorflow version: {tensorflow.__version__}")
print(f"numpy version: {numpy.__version__}")

pandas version: 2.2.2
joblib version: 1.5.3
scikit-learn version: 1.6.1
tensorflow version: 2.19.0
numpy version: 2.0.2
